# Air Passengers Time Series Forecasting with Simple RNN

## Objectives
Classic **Air Passengers** monthly series — forecast with a vanilla **SimpleRNN**.

## Theory
Vanilla **RNN** is simpler than LSTM: $h_t = \tanh(W x_t + U h_{t-1} + b)$. It works on smooth seasonal series but may struggle with very long dependencies compared to LSTM/GRU.


In [ ]:
# Optional: install dependencies (uncomment if needed)
# !pip install -q numpy pandas matplotlib seaborn scikit-learn tensorflow requests yfinance

import warnings
warnings.filterwarnings("ignore")

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
print("TensorFlow:", tf.__version__)


## Business Context
Airlines and tourism use passenger forecasts for **capacity planning** (routes, staff, fuel).


In [ ]:
# Built-in statsmodels dataset via pandas URL
url = "https://raw.githubusercontent.com/plotly/datasets/master/air-passengers.csv"
df = pd.read_csv(url)
df.columns = ["Month", "Passengers"]
df["Month"] = pd.to_datetime(df["Month"])
df = df.set_index("Month")
print(df.head())


In [ ]:
df.plot(title="Air Passengers")
plt.show()
from statsmodels.tsa.seasonal import seasonal_decompose
decomp = seasonal_decompose(df["Passengers"], model="multiplicative", period=12)
decomp.plot()
plt.show()


In [ ]:
series = df["Passengers"].values.astype(np.float32)
scaler = MinMaxScaler()
scaled = scaler.fit_transform(series.reshape(-1,1)).flatten()
lookback = 12

def make_seq(s, lb):
    X, y = [], []
    for i in range(lb, len(s)):
        X.append(s[i-lb:i])
        y.append(s[i])
    return np.array(X), np.array(y)

X, y = make_seq(scaled, lookback)
X = X[..., np.newaxis]
split = int(len(X)*0.8)
X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]
val_idx = int(0.85 * len(X_train))
X_tr, y_tr = X_train[:val_idx], y_train[:val_idx]
X_val, y_val = X_train[val_idx:], y_train[val_idx:]


In [ ]:
model = models.Sequential([
    layers.SimpleRNN(32, input_shape=(lookback, 1)),
    layers.Dense(1),
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()


In [ ]:
air_cb = [
    callbacks.ModelCheckpoint("rnn_air_best.keras", save_best_only=True),
    callbacks.EarlyStopping(patience=10, restore_best_weights=True),
]
history = model.fit(X_tr, y_tr, epochs=80, batch_size=8, verbose=1,
          validation_data=(X_val, y_val), callbacks=air_cb)
pd.DataFrame(history.history)["loss"].plot()
plt.title("Air Passengers RNN loss")
plt.show()


In [ ]:
pred = scaler.inverse_transform(model.predict(X_test, verbose=0))
actual = scaler.inverse_transform(y_test.reshape(-1,1))
print("RMSE:", np.sqrt(mean_squared_error(actual, pred)))
plt.plot(actual, label="Actual")
plt.plot(pred, label="RNN Pred")
plt.legend()
plt.show()


In [ ]:
# Inference — next month passengers
w = scaled[-lookback:].reshape(1, lookback, 1)
nxt = scaler.inverse_transform(model.predict(w, verbose=0))[0,0]
print(f"Next-month passenger forecast: {nxt:.0f}")
model.save("rnn_air_passengers.keras")


## Deployment Notes

1. **Serving**: Export with `model.export("saved_model")` for TensorFlow Serving, or wrap `predict` in FastAPI/Flask.
2. **Preprocessing**: Always apply the **same** scaler/encoder fitted on training data (`scaler.pkl`).
3. **Monitoring**: Track input drift, latency, and prediction distribution on live traffic.
4. **Retraining**: Schedule periodic retrain when performance drops below SLA.
5. **Security**: Do not log PII; use HTTPS and auth on inference endpoints.
